# Aula 2 — NumPy: arrays, matrizes e operações vetorizadas

**Módulo DS-PY-004 — Técnicas de Programação I (Python)**

Objetivos: entender por que o NumPy existe, criar e manipular arrays de 1 e N dimensões, indexar, fatiar, filtrar e operar de forma vetorizada.

Regra do notebook: **não use laços `for`** a não ser quando o enunciado pedir explicitamente. Todo o resto deve ser resolvido de forma vetorizada.

In [ ]:
import numpy as np
import time

np.random.seed(42)
print("NumPy:", np.__version__)

## Parte 1 — Por que NumPy? (lista vs. array)

### Exercício 1.1
Crie uma lista Python com os números de 0 a 9 e um array NumPy com os mesmos valores. Imprima o tipo (`type`) de cada um.

In [ ]:
lista = list(range(10))
arr = np.arange(10)
print(lista, type(lista))
print(arr, type(arr))

### Exercício 1.2
Tente multiplicar a **lista** por 2 e o **array** por 2. Explique em uma frase por que os resultados são diferentes.

In [ ]:
print(lista * 2)   # repete a lista (concatenação)
print(arr * 2)     # multiplica elemento a elemento (vetorização)
# A lista é uma sequência genérica: `*` significa repetição.
# O array é um bloco homogêneo de números: `*` é uma operação matemática elemento a elemento.

### Exercício 1.3
Compare o tempo necessário para elevar ao quadrado 5 milhões de números usando (a) list comprehension e (b) um array NumPy. Quantas vezes o NumPy foi mais rápido?

In [ ]:
n = 5_000_000
lista_grande = list(range(n))
array_grande = np.arange(n)

t0 = time.time()
_ = [x ** 2 for x in lista_grande]
t_lista = time.time() - t0

t0 = time.time()
_ = array_grande ** 2
t_numpy = time.time() - t0

print(f"lista: {t_lista:.4f}s | numpy: {t_numpy:.4f}s | ganho: {t_lista / t_numpy:.1f}x")

## Parte 2 — Criação e atributos

### Exercício 2.1
Crie, usando a função apropriada do NumPy (sem digitar os números um a um):

a) um array com os inteiros de 10 a 50 (inclusive)
b) um array com 9 números igualmente espaçados entre 0 e 2
c) um array 3x4 só de zeros
d) um array 2x5 só de uns, do tipo inteiro
e) uma matriz identidade 4x4
f) um array 3x3 com números aleatórios entre 0 e 1

In [ ]:
a = np.arange(10, 51)
b = np.linspace(0, 2, 9)
c = np.zeros((3, 4))
d = np.ones((2, 5), dtype=int)
e = np.eye(4)
f = np.random.rand(3, 3)

for nome, x in zip('abcdef', [a, b, c, d, e, f]):
    print(nome, '->', x.shape, x.dtype)
    print(x, '\n')

### Exercício 2.2
Para o array `dados` abaixo, imprima: número de dimensões, formato, quantidade total de elementos, tipo dos dados e quantos bytes ele ocupa na memória.

In [ ]:
dados = np.array([[1.5, 2.0, 3.5, 4.0],
                  [5.0, 6.5, 7.0, 8.5],
                  [9.0, 10.5, 11.0, 12.5]])

In [ ]:
print("ndim :", dados.ndim)
print("shape:", dados.shape)
print("size :", dados.size)
print("dtype:", dados.dtype)
print("bytes:", dados.nbytes)

### Exercício 2.3
Converta o array `dados` para o tipo inteiro. O que aconteceu com as casas decimais — houve arredondamento ou truncamento?

In [ ]:
print(dados.astype(int))
# Houve truncamento (a parte decimal é descartada, não arredondada):
# 2.5 vira 2, e não 3. Para arredondar use np.round antes do astype.
print(np.round(dados).astype(int))

## Parte 3 — Indexação e fatiamento (1D)

Use o array `vendas` abaixo, que representa as vendas diárias de uma loja ao longo de 14 dias.

In [ ]:
vendas = np.array([120, 85, 340, 90, 76, 410, 380,
                   95, 110, 305, 60, 88, 420, 395])

### Exercício 3.1
Selecione: (a) o primeiro elemento, (b) o último elemento, (c) os 3 primeiros, (d) do 5º ao 10º dia, (e) todos os elementos de índice par, (f) o array invertido.

In [ ]:
print("a:", vendas[0])
print("b:", vendas[-1])
print("c:", vendas[:3])
print("d:", vendas[4:10])
print("e:", vendas[::2])
print("f:", vendas[::-1])

### Exercício 3.2
Sabendo que o array começa numa segunda-feira, selecione apenas os finais de semana (sábados e domingos, isto é, o 6º e o 7º dia de cada semana).

In [ ]:
fds = vendas.reshape(2, 7)[:, 5:].ravel()
print(fds)
# Alternativa com índices explícitos:
print(vendas[[5, 6, 12, 13]])

### Exercício 3.3
Usando **máscara booleana**, retorne:
a) os dias com vendas acima de 300
b) quantos dias tiveram vendas abaixo de 100
c) os dias com vendas entre 100 e 350 (inclusive)

In [ ]:
print("a:", vendas[vendas > 300])
print("b:", (vendas < 100).sum())
print("c:", vendas[(vendas >= 100) & (vendas <= 350)])

### Exercício 3.4
Descubra em **quais posições** (índices) ocorreram as vendas acima de 300.

In [ ]:
print(np.where(vendas > 300)[0])

### Exercício 3.5
Crie um novo array em que toda venda abaixo de 100 seja substituída por 0 e as demais sejam mantidas. Garanta que o array `vendas` original **não** seja alterado.

In [ ]:
vendas_ajustadas = np.where(vendas < 100, 0, vendas)
print(vendas_ajustadas)
print(vendas)  # original intacto

### Exercício 3.6 (atenção — pegadinha clássica)
Crie `fatia = vendas[:3]` e altere `fatia[0] = 999`. Verifique o array `vendas`. O que aconteceu? Como evitar esse comportamento?

In [ ]:
vendas_teste = vendas.copy()
fatia = vendas_teste[:3]
fatia[0] = 999
print(vendas_teste[:5])
# O fatiamento no NumPy retorna uma VIEW (janela sobre a mesma memória), não uma cópia.
# Alterar a fatia altera o array original. Para evitar: vendas_teste[:3].copy()

## Parte 4 — Estatísticas e operações

### Exercício 4.1
Calcule para `vendas`: soma, média, mediana, desvio padrão, mínimo, máximo, e os índices do dia de menor e de maior venda.

In [ ]:
print("soma    :", vendas.sum())
print("média   :", vendas.mean())
print("mediana :", np.median(vendas))
print("desvio  :", vendas.std())
print("mínimo  :", vendas.min(), "no dia", vendas.argmin() + 1)
print("máximo  :", vendas.max(), "no dia", vendas.argmax() + 1)

### Exercício 4.2
Normalize o array `vendas` para a escala 0–1 (min-max):  (x - min) / (max - min). Confirme que o resultado vai de 0 a 1.

In [ ]:
norm = (vendas - vendas.min()) / (vendas.max() - vendas.min())
print(np.round(norm, 3))
print(norm.min(), norm.max())

### Exercício 4.3
Padronize `vendas` (z-score): (x - média) / desvio padrão. Quais dias estão a mais de 1 desvio padrão acima da média?

In [ ]:
z = (vendas - vendas.mean()) / vendas.std()
print(np.round(z, 2))
print("Dias:", np.where(z > 1)[0] + 1)

### Exercício 4.4
Calcule o **acumulado** de vendas ao longo dos 14 dias e descubra em que dia a loja ultrapassou 1000 de faturamento acumulado.

In [ ]:
acum = np.cumsum(vendas)
print(acum)
print("Ultrapassou 1000 no dia:", np.argmax(acum > 1000) + 1)

## Parte 5 — Arrays n-dimensionais (matrizes)

Considere a matriz `notas` abaixo: cada **linha** é um aluno e cada **coluna** é uma disciplina (Matemática, Estatística, Programação, Inglês).

In [ ]:
notas = np.array([[8.0, 7.5, 9.0, 6.0],
                  [5.5, 6.0, 7.0, 8.5],
                  [9.5, 9.0, 8.0, 7.5],
                  [4.0, 5.0, 6.5, 5.5],
                  [7.0, 8.5, 9.5, 9.0]])
alunos = np.array(['Ana', 'Bruno', 'Carla', 'Diego', 'Elena'])
disciplinas = np.array(['Matemática', 'Estatística', 'Programação', 'Inglês'])

### Exercício 5.1
Selecione: (a) a nota da Carla em Programação, (b) todas as notas do Bruno, (c) todas as notas de Inglês, (d) a submatriz com os 3 primeiros alunos e as 2 primeiras disciplinas.

In [ ]:
print("a:", notas[2, 2])
print("b:", notas[1])
print("c:", notas[:, 3])
print("d:\n", notas[:3, :2])

### Exercício 5.2
Calcule a média **por aluno** e a média **por disciplina**. Imprima cada uma junto com o nome correspondente.

*Dica: pense em qual eixo (`axis`) deve ser colapsado.*

In [ ]:
media_aluno = notas.mean(axis=1)
media_disc = notas.mean(axis=0)

for nome, m in zip(alunos, media_aluno):
    print(f"{nome:8s} {m:.2f}")
print()
for nome, m in zip(disciplinas, media_disc):
    print(f"{nome:12s} {m:.2f}")

### Exercício 5.3
Quem foi o aluno com a maior média? E qual foi a disciplina com a menor média da turma?

In [ ]:
print("Melhor média:", alunos[media_aluno.argmax()], round(media_aluno.max(), 2))
print("Pior disciplina:", disciplinas[media_disc.argmin()], round(media_disc.min(), 2))

### Exercício 5.4
Crie um array booleano indicando quais alunos foram aprovados (média >= 7). Depois imprima apenas os nomes dos aprovados e a taxa de aprovação da turma.

In [ ]:
aprovados = media_aluno >= 7
print(aprovados)
print("Aprovados:", alunos[aprovados])
print(f"Taxa de aprovação: {aprovados.mean():.0%}")

### Exercício 5.5 — broadcasting
A coordenação decidiu aplicar um bônus **diferente por disciplina**: +1.0 em Matemática, +0.5 em Estatística, 0 em Programação e +1.5 em Inglês. Aplique o bônus a todos os alunos de uma só vez, garantindo que nenhuma nota ultrapasse 10.

In [ ]:
bonus = np.array([1.0, 0.5, 0.0, 1.5])
notas_bonus = np.minimum(notas + bonus, 10)
print(notas_bonus)

### Exercício 5.6
Explique, em uma frase, por que a operação do exercício anterior funcionou mesmo com `notas` sendo 5x4 e `bonus` sendo 1x4. O que aconteceria se `bonus` tivesse 5 elementos?

In [ ]:
# Broadcasting: o NumPy compara os shapes da direita para a esquerda. Como a última
# dimensão bate (4 == 4), o array `bonus` é replicado virtualmente para cada uma das
# 5 linhas, sem cópia real na memória.
try:
    notas + np.array([1, 2, 3, 4, 5])
except ValueError as erro:
    print("Erro:", erro)
# Com 5 elementos, a última dimensão (4) não bate com 5 e o NumPy levanta ValueError.

## Parte 6 — Reshape e álgebra linear

### Exercício 6.1
Crie um array com os números de 1 a 12 e transforme-o em uma matriz 3x4. Depois transforme a mesma sequência em 4x3 e em 2x6. O que a função `reshape(-1, 2)` faz?

In [ ]:
base = np.arange(1, 13)
print(base.reshape(3, 4), '\n')
print(base.reshape(4, 3), '\n')
print(base.reshape(2, 6), '\n')
print(base.reshape(-1, 2))
# O -1 diz ao NumPy para inferir automaticamente essa dimensão (aqui, 6 linhas).

### Exercício 6.2
Dada a matriz `notas`, obtenha sua transposta e confirme o novo shape. Em que situação prática isso é útil?

In [ ]:
print(notas.T)
print(notas.shape, '->', notas.T.shape)
# Útil quando a orientação dos dados não bate com o que a função/modelo espera
# (ex.: virar "uma linha por observação" em "uma linha por variável").

### Exercício 6.3
Empilhe as matrizes `A` e `B` abaixo (a) verticalmente e (b) horizontalmente.

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

In [ ]:
print(np.vstack([A, B]), '\n')
print(np.hstack([A, B]))
# Equivalentes: np.concatenate([A, B], axis=0) e np.concatenate([A, B], axis=1)

### Exercício 6.4
Com `A` e `B`, calcule: (a) a multiplicação elemento a elemento, (b) o produto matricial, (c) o determinante de `A`, (d) a inversa de `A`. Verifique que `A @ inv(A)` resulta na identidade.

In [ ]:
print("a:\n", A * B)
print("b:\n", A @ B)
print("c:", np.linalg.det(A))
inv_A = np.linalg.inv(A)
print("d:\n", inv_A)
print("check:\n", np.round(A @ inv_A, 10))

### Exercício 6.5 — aplicação
Uma loja vende 3 produtos. A matriz `qtd` traz a quantidade vendida por produto em 4 dias e o vetor `precos` traz o preço unitário de cada produto. Calcule o faturamento diário usando produto matricial e depois o faturamento total.

In [ ]:
qtd = np.array([[10, 5, 2],
                [8, 7, 3],
                [12, 4, 1],
                [6, 9, 5]])
precos = np.array([25.0, 40.0, 120.0])

In [ ]:
faturamento_diario = qtd @ precos
print("Por dia :", faturamento_diario)
print("Total   :", faturamento_diario.sum())

## Parte 7 — Desafios

### Desafio 7.1
Gere 1000 valores aleatórios de uma distribuição normal com média 50 e desvio 10. Depois:
- confirme a média e o desvio obtidos
- conte quantos valores caem fora do intervalo [média - 2·desvio, média + 2·desvio]
- calcule que percentual isso representa (o esperado teórico é ~5%)

In [ ]:
amostra = np.random.normal(50, 10, 1000)
m, s = amostra.mean(), amostra.std()
print(f"média={m:.2f} desvio={s:.2f}")

fora = (amostra < m - 2 * s) | (amostra > m + 2 * s)
print("fora de 2 desvios:", fora.sum(), f"({fora.mean():.2%})")

### Desafio 7.2
Crie uma matriz 5x5 com valores aleatórios inteiros entre 1 e 100 e, **sem usar laços**:
- substitua todos os valores da diagonal principal por 0
- normalize cada **coluna** para que sua soma seja 1
- confirme que a soma de cada coluna é 1

In [ ]:
M = np.random.randint(1, 101, size=(5, 5))
print(M, '\n')

np.fill_diagonal(M, 0)
print(M, '\n')

M_norm = M / M.sum(axis=0)
print(np.round(M_norm, 3), '\n')
print("somas por coluna:", M_norm.sum(axis=0))

### Desafio 7.3
Dado o array com valores faltantes representados por `np.nan` abaixo:
- conte quantos valores estão faltando
- calcule a média ignorando os `nan`
- substitua os `nan` por essa média

In [ ]:
com_nan = np.array([12.0, np.nan, 15.0, 18.0, np.nan, 22.0, 9.0, 30.0])

In [ ]:
mascara = np.isnan(com_nan)
print("faltantes:", mascara.sum())

media_valida = np.nanmean(com_nan)
print("média ignorando nan:", round(media_valida, 2))

preenchido = np.where(mascara, media_valida, com_nan)
print(preenchido)

---
## Mini projeto NumPy (entrega em repositório Git)

Crie um notebook novo, em um repositório Git próprio, resolvendo o problema abaixo. **Nenhum laço `for` é permitido nos cálculos.**

**Contexto:** você recebeu a matriz de leituras de 6 sensores de temperatura ao longo de 30 dias (6 linhas x 30 colunas). Alguns sensores falharam em alguns dias e registraram `np.nan`.

Use o código abaixo para gerar os dados (mantenha a semente para que o resultado seja reproduzível):

```python
import numpy as np
rng = np.random.default_rng(2024)
leituras = rng.normal(24, 4, size=(6, 30)).round(1)
falhas = rng.random(size=(6, 30)) < 0.08
leituras[falhas] = np.nan
```

**Entregas:**
1. Quantas leituras faltantes existem no total? E por sensor?
2. Média, mediana, mínimo e máximo de cada sensor (ignorando os `nan`).
3. A temperatura média da estação em cada dia (média entre sensores).
4. Em quais dias a média diária ultrapassou 26 graus?
5. Identifique leituras atípicas: valores a mais de 2 desvios padrão da média geral. Quantas são e em quais sensores estão?
6. Preencha os `nan` com a **média do próprio sensor** e mostre que não restou nenhum faltante.
7. Normalize a matriz preenchida por sensor (z-score linha a linha) e apresente o resultado arredondado em 2 casas.
8. Escreva um parágrafo curto de conclusão: qual sensor parece menos confiável e por quê?

**Critérios de avaliação:** correção dos cálculos · uso de vetorização (sem laços) · legibilidade e comentários · repositório Git com README e histórico de commits coerente.